# Week 3 — Contract Similarity and a Nearest-Neighbor Pricing Baseline
## Overview

We now price contracts by transferring information from a small reference set. The guiding question is: **When do similar contract inputs imply similar normalized guarantee prices?** We begin with an ordinary distance and a nearest-reference predictor before trying to learn a better distance later in the project.

This week also establishes a clean experimental boundary. We split contract IDs into a candidate pool, a validation pool, and a final test pool before fitting transformations or generating model labels. We simulate only reference contracts from the candidate pool and an independent validation subset. The final test pool remains unpriced and unevaluated.

**By the end, you should be able to:**

- Explain the difference between similarity of input features and similarity of pricing responses.
- Fit min–max scaling on the candidate pool and apply the same transformation to queries.
- Implement a CSM-style one-nearest-reference price transfer with the correct premium normalization.
- Compare raw and scaled Euclidean distance using identical reference contracts and labels.
- Interpret validation scores in the presence of Monte Carlo label noise.
- Describe how $k$-nearest-neighbor averaging changes the predictor.

**Prerequisites:** Weeks 1–2; Euclidean distance; means and variance; table indexing. No clustering or metric-learning background is needed.

**Concrete outputs:** an explicit split summary; a premium-distance counterexample; one inspection of nearest references; a controlled raw-versus-scaled validation comparison; and an optional $k$-neighbor comparison.

**Time plan:** 4–6 independent hours plus a 75-minute meeting. Guided defaults use 1,200 contracts, 40 reference contracts, and 96 validation contracts. Their simulations should finish within a minute or two on a laptop.

**Target reminder:** normalized gross GMAB top-up at a specified fee. We assume survival to maturity with no survival-probability weighting. Increasing the fee lowers the account and weakly increases this gross top-up target. The core is a research comparison of pricing predictors, not a fair-fee root-finding algorithm.

### Setup and complete reference functions

The next code cells include everything this notebook needs. Run them once; they do not import a sibling notebook or download data. The simulator follows Week 2's plain independent-draw convention. Today's main learning functions are `build_features`, `select_representatives`, `nearest_predict`, and `score_predictions`; other reference helpers will be used in later weeks.

The full toolbox includes an analytical audit function for consistency across notebooks, but **no worked cell in this notebook calls it**. Validation prices are independently simulated.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from time import perf_counter
try:
    from IPython.display import display
except ImportError:
    def display(*objects):
        for obj in objects:
            print(obj)

pd.set_option("display.precision", 6)
r = 0.03

In [ ]:
# Complete common pricing code. No data files or earlier notebooks are required.
import sys
import json
import platform
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm

try:
    from IPython.display import display
except ImportError:
    def display(*objects):
        for obj in objects:
            print(obj)

RISK_FREE_RATE = 0.03
FEATURES = ['account_ratio', 'guarantee_ratio', 'maturity', 'volatility', 'fee']
plt.rcParams.update({'figure.figsize': (7.2, 4.3), 'axes.grid': True,
                     'grid.alpha': 0.22, 'font.size': 10})


def make_portfolio(n=1200, seed=2027):
    """Synthetic in-force GMAB contracts, never customer data.

    P is a reference premium, A0=P*account_ratio, and G=P*guarantee_ratio.
    T is the remaining term. All contracts share the same risk-free rate.
    Volatility describes the contract's fund; rates are annual decimals.
    """
    if not isinstance(n, (int, np.integer)) or n < 10:
        raise ValueError('n must be an integer of at least 10.')
    rng = np.random.default_rng(seed)
    return pd.DataFrame({
        'contract_id': np.arange(n, dtype=int),
        'premium': rng.uniform(10_000.0, 50_000.0, n),
        'account_ratio': rng.uniform(0.65, 1.40, n),
        'guarantee_ratio': rng.uniform(0.90, 1.60, n),
        'maturity': rng.integers(5, 21, n).astype(float),
        'volatility': rng.uniform(0.12, 0.35, n),
        'fee': rng.uniform(0.005, 0.030, n),
    })


def validate_contracts(df):
    required = ['contract_id', 'premium'] + FEATURES
    missing = [c for c in required if c not in df]
    if missing:
        raise ValueError(f'Missing contract columns: {missing}')
    if len(df) == 0:
        raise ValueError('Supply at least one contract.')
    values = df[required].to_numpy(dtype=float)
    if not np.isfinite(values).all():
        raise ValueError('All contract fields must be finite.')
    ids = df['contract_id'].to_numpy(dtype=float)
    if (ids < 0).any() or (ids != np.floor(ids)).any() or len(np.unique(ids)) != len(ids):
        raise ValueError('contract_id must contain unique nonnegative integers.')
    if (df[['premium', 'account_ratio', 'guarantee_ratio']] <= 0).any().any():
        raise ValueError('Premium and account/guarantee ratios must be positive.')
    if (df[['maturity', 'volatility', 'fee']] < 0).any().any():
        raise ValueError('Maturity, volatility and fee must be nonnegative.')


def gmab_mc(df, n_paths=2048, seed=123):
    """Independent plain Monte Carlo for per-premium gross top-up liability.

    Identical (seed, contract_id) reproduces the same path stream even when
    the contract appears in a different subset or row order. Distinct IDs
    give independent streams. This is NOT a common-market portfolio-risk
    simulator: it estimates a sum of expected liabilities, not portfolio VaR.
    SE describes Monte Carlo sampling uncertainty only.
    """
    validate_contracts(df)
    if not isinstance(n_paths, (int, np.integer)) or n_paths < 2:
        raise ValueError('n_paths must be an integer of at least 2.')
    if not isinstance(seed, (int, np.integer)) or seed < 0:
        raise ValueError('seed must be a nonnegative integer.')
    values, errors, sds = [], [], []
    for row in df.itertuples():
        rng = np.random.default_rng(np.random.SeedSequence([int(seed), int(row.contract_id)]))
        z = rng.standard_normal(n_paths)
        terminal_ratio = row.account_ratio * np.exp(
            (RISK_FREE_RATE - row.fee - 0.5 * row.volatility**2) * row.maturity
            + row.volatility * np.sqrt(row.maturity) * z
        )
        payoff = np.exp(-RISK_FREE_RATE * row.maturity) * np.maximum(
            row.guarantee_ratio - terminal_ratio, 0.0)
        sd = float(np.std(payoff, ddof=1))
        values.append(float(np.mean(payoff)))
        errors.append(sd / np.sqrt(n_paths))
        sds.append(sd)
    return pd.DataFrame({'value': values, 'se': errors, 'payoff_sd': sds,
                         'paths': int(n_paths)}, index=df.index)


def gmab_exact(df):
    """Closed-form put benchmark under the simplified model: audit only.

    Never use these values as training or tuning labels in the project.
    The deterministic cases T=0 or sigma=0 are handled separately.
    """
    validate_contracts(df)
    a = df['account_ratio'].to_numpy(dtype=float)
    g = df['guarantee_ratio'].to_numpy(dtype=float)
    t = df['maturity'].to_numpy(dtype=float)
    sig = df['volatility'].to_numpy(dtype=float)
    fee = df['fee'].to_numpy(dtype=float)
    ans = np.maximum(g * np.exp(-RISK_FREE_RATE*t) - a * np.exp(-fee*t), 0.0)
    regular = (t > 0) & (sig > 0)
    if np.any(regular):
        tr, sr = t[regular], sig[regular]
        d1 = (np.log(a[regular] / g[regular]) +
              (RISK_FREE_RATE-fee[regular]+0.5*sr**2)*tr) / (sr*np.sqrt(tr))
        d2 = d1 - sr*np.sqrt(tr)
        ans[regular] = (g[regular]*np.exp(-RISK_FREE_RATE*tr)*norm.cdf(-d2)
                        - a[regular]*np.exp(-fee[regular]*tr)*norm.cdf(-d1))
    return np.maximum(ans, 0.0)


def split_portfolio(df, seed=2027):
    """Split contracts before building pairs, selecting labels or tuning."""
    validate_contracts(df)
    rng = np.random.default_rng(seed)
    order = rng.permutation(len(df))
    n_candidate, n_validation = int(0.60*len(df)), int(0.20*len(df))
    if min(n_candidate, n_validation, len(df)-n_candidate-n_validation) < 1:
        raise ValueError('Portfolio is too small for the requested split.')
    return {
        'candidate': df.iloc[order[:n_candidate]].copy(),
        'validation': df.iloc[order[n_candidate:n_candidate+n_validation]].copy(),
        'test': df.iloc[order[n_candidate+n_validation:]].copy(),
    }


def score_predictions(y_true, y_pred, premium=None):
    """Errors per premium, plus signed aggregate relative error when requested."""
    truth = np.asarray(y_true, dtype=float).reshape(-1)
    pred = np.asarray(y_pred, dtype=float).reshape(-1)
    if truth.shape != pred.shape or len(truth) == 0:
        raise ValueError('Truth and predictions must have equal nonzero length.')
    if not (np.isfinite(truth).all() and np.isfinite(pred).all()):
        raise ValueError('Truth and predictions must be finite.')
    error = pred-truth
    out = {'rmse': float(np.sqrt(np.mean(error**2))),
           'mae': float(np.mean(np.abs(error))),
           'p95_abs': float(np.quantile(np.abs(error), 0.95)),
           'bias': float(np.mean(error)),
           'negative_fraction': float(np.mean(pred < 0))}
    if premium is not None:
        p = np.asarray(premium, dtype=float).reshape(-1)
        if p.shape != truth.shape or not np.isfinite(p).all() or np.any(p <= 0):
            raise ValueError('Premium weights must be finite, positive and aligned.')
        denominator = float(p @ truth)
        out['portfolio_relative_error'] = (float(p @ error)/denominator
                                            if abs(denominator) > 1e-12 else np.nan)
    return out


In [ ]:
# Complete shared learning code. All labels are supplied explicitly by the caller.
from scipy.spatial.distance import cdist
from scipy.optimize import minimize
from sklearn.preprocessing import MinMaxScaler, SplineTransformer
from sklearn.cluster import KMeans
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.pipeline import make_pipeline


def build_features(candidate, query=None, columns=FEATURES):
    """Fit feature scales only on the unlabelled candidate pool.

    A constant training column maps to zero on training data. A held-out
    observation outside the training range may map outside [0,1]; no clipping.
    No premium, contract ID or computed valuation is an informative feature.
    """
    scaler = MinMaxScaler()
    z_candidate = scaler.fit_transform(candidate[list(columns)])
    z_query = None if query is None else scaler.transform(query[list(columns)])
    return scaler, z_candidate, z_query


def select_representatives(Z, m, method='kmeans', seed=2027):
    """Return m DISTINCT row positions of actual candidate contracts.

    K-means is fitted to candidate FEATURES only. For each nonempty cluster,
    choose the member closest to that cluster's center. This is a member
    representative, not an exact k-medoids optimization or Feng et al.'s SRSC.
    """
    z = np.asarray(Z, dtype=float)
    if z.ndim != 2 or not np.isfinite(z).all():
        raise ValueError('Z must be a finite 2D matrix.')
    if not isinstance(m, (int, np.integer)) or not 1 <= m <= len(z):
        raise ValueError('m must lie between 1 and the number of candidates.')
    rng = np.random.default_rng(seed)
    if method == 'random':
        return np.sort(rng.choice(len(z), size=m, replace=False))
    if method != 'kmeans':
        raise ValueError("method must be 'random' or 'kmeans'.")
    km = KMeans(n_clusters=m, n_init=5, max_iter=150, random_state=seed)
    membership = km.fit_predict(z)
    chosen = []
    for cluster in range(m):
        members = np.flatnonzero(membership == cluster)
        if len(members):
            distances = np.sum((z[members] - km.cluster_centers_[cluster])**2, axis=1)
            chosen.append(int(members[np.argmin(distances)]))
    # Duplicate feature vectors can yield fewer than m nonempty clusters.
    # Keep the requested contract count without ever repeating an ID.
    if len(chosen) < m:
        remaining = np.setdiff1d(np.arange(len(z)), chosen)
        chosen.extend(rng.choice(remaining, m-len(chosen), replace=False).tolist())
    return np.sort(np.asarray(chosen, dtype=int))


def nearest_predict(Zref, yref, Zquery, weights=None, k=1):
    """Uniform k-neighbor transfer; k=1 is the CSM-style teaching baseline.

    A diagonal weighted distance is ordinary Euclidean distance after each
    coordinate is multiplied by sqrt(weight). Stable sort breaks exact ties
    in reference-row order. Chunking avoids an N-by-N distance matrix.
    """
    zr, zq = np.asarray(Zref, dtype=float), np.asarray(Zquery, dtype=float)
    yr = np.asarray(yref, dtype=float).reshape(-1)
    if zr.ndim != 2 or zq.ndim != 2 or zr.shape[1] != zq.shape[1] or len(zr) != len(yr):
        raise ValueError('Reference/query dimensions or labels do not align.')
    if not (np.isfinite(zr).all() and np.isfinite(zq).all() and np.isfinite(yr).all()):
        raise ValueError('Features and labels must be finite.')
    if not isinstance(k, (int, np.integer)) or not 1 <= k <= len(zr):
        raise ValueError('k must be between 1 and the reference count.')
    w = np.ones(zr.shape[1]) if weights is None else np.asarray(weights, dtype=float)
    if w.shape != (zr.shape[1],) or not np.isfinite(w).all() or np.any(w < 0):
        raise ValueError('Distance weights must be finite and nonnegative.')
    scale = np.sqrt(w)
    neighbors = np.empty((len(zq), k), dtype=int)
    for start in range(0, len(zq), 1024):
        stop = min(start+1024, len(zq))
        distance = cdist(zq[start:stop]*scale, zr*scale, metric='sqeuclidean')
        neighbors[start:stop] = np.argsort(distance, axis=1, kind='stable')[:, :k]
    return yr[neighbors].mean(axis=1), neighbors


def fit_metric(Zref, yref, lam=0.01, max_pairs=2000, seed=2027):
    """Fit a simple diagonal response-aware pseudometric on reference pairs.

    Objective: mean(((H@w)/p - ((y_i-y_j)/s)^2)^2)
               + lam/2 * sum((w-1)^2),  w>=0, sum(w)=p.
    Here H=(z_i-z_j)^2 coordinatewise and s=max(ptp(yref),1e-6).
    The label scale is part of the modeling choice, fitted on references only.
    Numerical labels have noise: this loss is not an unbiased exact-gap loss.
    """
    z, y = np.asarray(Zref, dtype=float), np.asarray(yref, dtype=float).reshape(-1)
    if z.ndim != 2 or len(z) != len(y) or len(y) < 2:
        raise ValueError('Supply at least two aligned labeled reference contracts.')
    if not (np.isfinite(z).all() and np.isfinite(y).all()):
        raise ValueError('Training data must be finite.')
    if lam < 0 or not isinstance(max_pairs, (int, np.integer)) or max_pairs < 1:
        raise ValueError('lam must be nonnegative and max_pairs a positive integer.')
    n, p = z.shape
    count = min(max_pairs, n*(n-1)//2)
    rng = np.random.default_rng(seed)
    if count == n*(n-1)//2:
        i, j = np.triu_indices(n, k=1)
    else:
        # Uniform directed non-self pairs sampled WITH replacement.
        # They share contracts and are not independent observational units.
        i = rng.integers(0, n, size=count)
        j = (i+rng.integers(1, n, size=count)) % n
    label_scale = max(float(np.ptp(y)), 1e-6)
    h = (z[i]-z[j])**2 / p
    target = ((y[i]-y[j])/label_scale)**2
    def objective(w):
        residual = h @ w-target
        return float(np.mean(residual**2) + 0.5*lam*np.sum((w-1.0)**2))
    def gradient(w):
        return 2.0*h.T @ (h @ w-target)/len(target) + lam*(w-1.0)
    result = minimize(objective, np.ones(p), jac=gradient, method='SLSQP',
                      bounds=[(0.0, float(p))]*p,
                      constraints=[{'type': 'eq', 'fun': lambda w: w.sum()-p,
                                    'jac': lambda w: np.ones(p)}],
                      options={'maxiter': 400, 'ftol': 1e-10})
    feasible = (np.isfinite(result.x).all() and result.x.min() >= -1e-7
                and abs(result.x.sum()-p) <= 1e-6)
    success = bool(result.success and feasible)
    weights = np.clip(result.x, 0.0, None) if success else np.ones(p)
    weights *= p/weights.sum()
    return {'weights': weights, 'success': success, 'pair_count': len(target),
            'objective': objective(weights), 'scale': label_scale,
            'iterations': int(result.nit), 'message': str(result.message)}


def make_regressors(seed=2027):
    """Fixed small model menu. Regression predicts values, not distances."""
    return {
        'Ridge': Ridge(alpha=1.0),
        'SplineGAM': make_pipeline(
            SplineTransformer(n_knots=4, degree=3, include_bias=False),
            Ridge(alpha=1.0)),
        'RandomForest': RandomForestRegressor(n_estimators=100, min_samples_leaf=3,
                                               random_state=seed, n_jobs=1),
        'GradientBoosting': GradientBoostingRegressor(n_estimators=100, max_depth=2,
                                                       learning_rate=0.05, random_state=seed),
    }


## 1. Split contracts before preparing features or looking at labels

We distinguish three sets with different jobs:

| Pool | Default share | Permitted use |
|---|---:|---|
| Candidate | 60% | Fit feature scaling; choose representatives; obtain their simulation labels; fit a predictor |
| Validation | 20% | Compare or tune candidate choices using separately simulated labels |
| Test | 20% | One final assessment after the choices have been frozen, in Week 8, after Week 7 development and design lock |

A random split is by **contract ID**, not by individual Monte Carlo paths. Splitting paths from the same contract into training and test would not ask whether a predictor generalizes to new contracts.

We fit min–max scaling on the candidate features only. Validation and test data must not change the transformation. Merely knowing a contract's features does not reveal its price, but this discipline makes the role of each set unambiguous and carries over to richer preprocessing later.

The candidate pool contains many contracts without simulation labels. It is legitimate to use their inputs for the representative-selection stage. The predictor is trained only on the selected representatives' simulated labels.

Our split is reproducible but is not a claim that random held-out contracts cover every future market regime or product design. Distribution shift is a separate question.

In [ ]:
portfolio = make_portfolio(n=1200, seed=2027)
pools = split_portfolio(portfolio, seed=2027)
candidate = pools["candidate"].copy()
validation = pools["validation"].copy()
test = pools["test"].copy()

candidate_ids = set(candidate["contract_id"])
validation_ids = set(validation["contract_id"])
test_ids = set(test["contract_id"])
assert candidate_ids.isdisjoint(validation_ids)
assert candidate_ids.isdisjoint(test_ids)
assert validation_ids.isdisjoint(test_ids)
assert len(candidate_ids | validation_ids | test_ids) == len(portfolio)

split_summary = pd.DataFrame({
    "pool": ["candidate", "validation", "test"],
    "contracts": [len(candidate), len(validation), len(test)],
    "labels_generated_now": [False, False, False],
})
display(split_summary)
print("No test labels will be generated or evaluated in this notebook.")

## 2. Euclidean distance depends on what we measure and in which units

For inputs $x_i,x_j\in\mathbb R^p$, raw Euclidean distance is

$$d(x_i,x_j)=\sqrt{\sum_{\ell=1}^p(x_{i\ell}-x_{j\ell})^2}.$$

This calculation treats a numerical difference of `1` in any coordinate equally. In our data, maturity is measured in years while fees and volatility are decimal rates. Those numbers are not naturally comparable. Changing maturity from years to months changes raw distance even when the contract has not changed.

Premium presents an even more basic issue. Because our response is normalized by premium, two contracts with identical $(a,g,T,\sigma,\phi)$ have the same pricing target even when their premiums differ. The toy example below shows how including premium can select an inappropriate neighbor. This conclusion follows from the scaling identity in Week 1; it does not use analytical price labels.

In [ ]:
toy_contracts = pd.DataFrame({
    "name": ["Query", "Same premium, different shape", "Same shape, larger premium"],
    "premium": [20000.0, 20000.0, 80000.0],
    "account_ratio": [1.0, 0.9, 1.0],
    "guarantee_ratio": [1.0, 1.1, 1.0],
    "maturity": [10.0, 20.0, 10.0],
    "volatility": [0.20, 0.25, 0.20],
    "fee": [0.01, 0.015, 0.01],
})
toy_columns_with_premium = ["premium"] + FEATURES
toy_raw = toy_contracts[toy_columns_with_premium].to_numpy()
toy_shapes = toy_contracts[FEATURES].to_numpy()
toy_comparison = pd.DataFrame({
    "candidate_neighbor": toy_contracts["name"].iloc[1:].to_numpy(),
    "raw_distance_including_premium": np.linalg.norm(toy_raw[1:] - toy_raw[0], axis=1),
    "shape_distance_excluding_premium": np.linalg.norm(toy_shapes[1:] - toy_shapes[0], axis=1),
})
display(toy_contracts)
display(toy_comparison)
print("Identical normalized contract shapes have identical normalized targets in our model.")

### Min–max scaling gives every feature a comparable observed range

For feature $\ell$, let $m_\ell$ and $M_\ell$ be its minimum and maximum in the candidate pool. For a nonconstant feature, define

$$z_\ell(x)=\frac{x_\ell-m_\ell}{M_\ell-m_\ell}.$$

Candidate values then lie in $[0,1]$. The same fitted transformation is applied to every validation or future query. A query outside the candidate range may have a value below zero or above one; we do not silently refit or clip it.

We use scikit-learn's `MinMaxScaler` convention for constant candidate columns: the observed constant candidate values map to zero, and the library uses a unit denominator for subsequent transformations. The present synthetic portfolio varies all five features. In a real dataset, record and review constant columns instead of assuming they carry useful information.

Scaling makes the numerical units comparable. It does **not** establish that a full observed-range change in fee should matter exactly as much for price as a full observed-range change in volatility. That limitation motivates the learned-distance experiment in Week 6.

In [ ]:
# Reserve a reproducible validation subset for this week's modest demonstration.
validation_demo = validation.sample(n=min(96, len(validation)), random_state=30001).copy()

scaler, Zcandidate, Zvalidation = build_features(candidate, validation_demo)
raw_candidate = candidate[FEATURES].to_numpy()
raw_validation = validation_demo[FEATURES].to_numpy()

scaling_table = pd.DataFrame({
    "feature": FEATURES,
    "candidate_min": scaler.data_min_,
    "candidate_max": scaler.data_max_,
    "candidate_range": scaler.data_range_,
    "scaled_candidate_min": Zcandidate.min(axis=0),
    "scaled_candidate_max": Zcandidate.max(axis=0),
    "scaled_validation_min": Zvalidation.min(axis=0),
    "scaled_validation_max": Zvalidation.max(axis=0),
})
display(scaling_table)
assert Zcandidate.shape == (len(candidate), len(FEATURES))
assert Zvalidation.shape == (len(validation_demo), len(FEATURES))

## 3. The nearest-reference predictor: transfer a simulated normalized price

Select $m$ actual contracts from the candidate pool. Write their transformed inputs as $z_1,\ldots,z_m$ and their Monte Carlo prices as $\widetilde y_1,\ldots,\widetilde y_m$.

For a query contract $x$, choose

$$j^*(x)=\arg\min_{1\leq j\leq m}\|z(x)-z_j\|_2,$$

and predict

$$\widehat y(x)=\widetilde y_{j^*(x)},\qquad
\widehat L(x)=P_x\,\widetilde y_{j^*(x)}.$$

This is our **CSM-style one-nearest-reference baseline**, inspired by the representative-based valuation approach in Feng, Tan, and Zheng (2020). We transfer the reference's liability **per unit of premium**, then multiply by the query premium. Copying the reference's dollar liability would confound contract shape and contract size. We use a simplified product and uniform simulation budgets; this is not a complete reproduction of that paper's simulation design.

There are two distinct error sources:

$$\widehat y(x)-y(x)
=\underbrace{\widetilde y_{j^*}-y(x_{j^*})}_{\text{reference simulation noise}}
+\underbrace{y(x_{j^*})-y(x)}_{\text{price mismatch between contracts}}.$$

More paths can reduce the first term. Better reference coverage or a better similarity rule may reduce the second. More paths do not automatically make a poorly matched neighbor appropriate.

We begin with random representatives, chosen without looking at prices. Clustering-based selection is next week's topic.

In [ ]:
n_representatives = 40
reference_paths = 4096
validation_paths = 8192

selection_start = perf_counter()
reference_positions = select_representatives(
    Zcandidate, m=n_representatives, method="random", seed=31001
)
selection_seconds = perf_counter() - selection_start
reference_contracts = candidate.iloc[reference_positions].copy()
Zreference = Zcandidate[reference_positions]
raw_reference = raw_candidate[reference_positions]

reference_start = perf_counter()
reference_labels = gmab_mc(reference_contracts, n_paths=reference_paths, seed=32001)
reference_seconds = perf_counter() - reference_start
y_reference = reference_labels["value"].to_numpy()

validation_start = perf_counter()
validation_labels = gmab_mc(validation_demo, n_paths=validation_paths, seed=33001)
validation_seconds = perf_counter() - validation_start
y_validation_mc = validation_labels["value"].to_numpy()

assert set(reference_contracts["contract_id"]).issubset(candidate_ids)
assert set(reference_contracts["contract_id"]).isdisjoint(set(validation_demo["contract_id"]))
assert len(set(reference_positions)) == n_representatives

display(reference_contracts.head())
display(reference_labels.head())
print(f"Simulated {len(reference_contracts)} candidate references and "
      f"{len(validation_demo)} independent validation contracts.")

In [ ]:
raw_start = perf_counter()
pred_raw, neighbor_raw = nearest_predict(raw_reference, y_reference, raw_validation, k=1)
raw_prediction_seconds = perf_counter() - raw_start

scaled_start = perf_counter()
pred_scaled, neighbor_scaled = nearest_predict(Zreference, y_reference, Zvalidation, k=1)
scaled_prediction_seconds = perf_counter() - scaled_start

# Inspect which actual reference is selected for the same query by each distance.
query_position = 0
raw_reference_position = int(neighbor_raw[query_position, 0])
scaled_reference_position = int(neighbor_scaled[query_position, 0])
inspection_table = pd.DataFrame([
    validation_demo.iloc[query_position].to_dict(),
    reference_contracts.iloc[raw_reference_position].to_dict(),
    reference_contracts.iloc[scaled_reference_position].to_dict(),
], index=["Validation query", "Raw-distance reference", "Scaled-distance reference"])
display(inspection_table)

prediction_inspection = pd.DataFrame({
    "method": ["Raw Euclidean 1NN", "Scaled Euclidean 1NN"],
    "predicted_value_per_premium": [pred_raw[query_position], pred_scaled[query_position]],
    "predicted_liability_dollars": [
        validation_demo.iloc[query_position]["premium"] * pred_raw[query_position],
        validation_demo.iloc[query_position]["premium"] * pred_scaled[query_position],
    ],
})
display(prediction_inspection)

**Interpretation checkpoint.** Look at the selected references rather than only their IDs. Which feature differences are large in the original units? Which are large relative to the candidate ranges? The two rules may choose the same reference for some queries; do not alter the data merely to force a difference.

## 4. Compare predictors against independent Monte Carlo validation labels

Let $\widetilde y_i^{\mathrm{val}}$ be a high-path, independently simulated validation label. For $n_v$ validation contracts, we report

$$\operatorname{RMSE}_{\mathrm{MC}}=
\sqrt{\frac{1}{n_v}\sum_i(\widehat y_i-\widetilde y_i^{\mathrm{val}})^2},
\qquad
\operatorname{MAE}_{\mathrm{MC}}=\frac{1}{n_v}\sum_i|\widehat y_i-\widetilde y_i^{\mathrm{val}}|.$$

We also report the 95th percentile of absolute errors and the signed mean error. For these per-contract metrics, multiplying by 100 expresses the result in **percentage points of premium**.

The labels are estimates, not exact truths. For a predictor fixed before inspecting independent validation noise, write $\widetilde y_i^{\mathrm{val}}=y_i+\eta_i$, with $\mathbb E[\eta_i]=0$. Then

$$\mathbb E_{\eta_i}\!\left[(\widehat y_i-\widetilde y_i^{\mathrm{val}})^2\mid\widehat y_i\right]
=(\widehat y_i-y_i)^2+\operatorname{Var}(\eta_i).$$

Thus the expected squared discrepancy includes validation simulation noise. The root-mean-square reported validation standard error is a useful scale to compare with the RMSE; it is not a confidence interval for the model comparison. A small observed difference between models may be too noisy to interpret. If we choose a winner by inspecting these scores, that selection creates an additional reason to reserve a separate final test set.

The signed portfolio relative error uses dollar weights:

$$\frac{\sum_i P_i\widehat y_i-\sum_i P_i\widetilde y_i^{\mathrm{val}}}
{\sum_i P_i\widetilde y_i^{\mathrm{val}}}.$$

It can be small because positive and negative individual errors cancel. Always inspect individual-contract error metrics as well. In this notebook it describes the validation subset only, not the full portfolio.

In [ ]:
validation_premiums = validation_demo["premium"].to_numpy()
validation_predictions = {
    "Raw Euclidean 1NN": pred_raw,
    "Scaled Euclidean 1NN": pred_scaled,
}
validation_score_rows = []
for name, predictions in validation_predictions.items():
    metrics = score_predictions(y_validation_mc, predictions, premium=validation_premiums)
    validation_score_rows.append({
        "method": name,
        "RMSE_pp_of_premium": 100 * metrics["rmse"],
        "MAE_pp_of_premium": 100 * metrics["mae"],
        "P95_absolute_error_pp": 100 * metrics["p95_abs"],
        "signed_bias_pp": 100 * metrics["bias"],
        "validation_total_relative_error_percent": 100 * metrics["portfolio_relative_error"],
    })
validation_scores = pd.DataFrame(validation_score_rows)
validation_noise_rms = float(np.sqrt(np.mean(validation_labels["se"].to_numpy()**2)))
display(validation_scores)
print(f"RMS validation-label SE: {100 * validation_noise_rms:.4f} percentage points of premium")
print("Scores compare predictions with independent Monte Carlo labels, not exact prices.")

In [ ]:
plot_upper = 100 * max(float(np.max(y_validation_mc)), float(np.max(pred_raw)),
                       float(np.max(pred_scaled)), 1e-6)
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), sharex=True, sharey=True)
for ax, (name, predictions) in zip(axes, validation_predictions.items()):
    ax.scatter(100 * y_validation_mc, 100 * predictions, s=25, alpha=0.7)
    ax.plot([0, plot_upper], [0, plot_upper], "--", color="gray", label="Equal values")
    ax.set(xlabel="Validation MC value (% of premium)",
           ylabel="Predicted value (% of premium)", title=name)
    ax.grid(alpha=0.25)
    ax.legend()
fig.suptitle("Validation only: identical references and simulation labels")
fig.tight_layout()
plt.show()

scaled_nearest_distances = np.linalg.norm(
    Zvalidation - Zreference[neighbor_scaled[:, 0]], axis=1
)
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(scaled_nearest_distances, 100 * np.abs(pred_scaled - y_validation_mc), alpha=0.7)
ax.set(xlabel="Distance to selected reference in scaled feature space",
       ylabel="Absolute discrepancy from validation MC (pp of premium)",
       title="Does input proximity reliably indicate pricing accuracy?")
ax.grid(alpha=0.25)
fig.tight_layout()
plt.show()

### Keep the comparison controlled and record its cost

Only the distance rule differs between the two main predictors. They use the same contracts, reference labels, path counts, and validation labels. Regenerating labels separately for each rule would add an avoidable simulation-noise difference.

We record representative selection, reference simulation, prediction, and validation simulation separately. Feature preparation has already been performed and is not included in this short timing demonstration; the complete Week 7 benchmark will time the whole workflow. These figures therefore should not be reported as end-to-end speedups.

Selection and reference simulation are shared by the two predictors in this experiment. The validation simulation is evaluation work and is also shared. Do not count the same physical work twice when reporting the total cost of running this notebook.

In [ ]:
cost_breakdown = pd.DataFrame({
    "stage": ["Random reference selection", "Shared reference simulation",
              "Raw 1NN prediction", "Scaled 1NN prediction", "Shared validation simulation"],
    "seconds": [selection_seconds, reference_seconds, raw_prediction_seconds,
                scaled_prediction_seconds, validation_seconds],
    "contract_path_payoffs": [0, int(reference_labels["paths"].sum()), 0, 0,
                              int(validation_labels["paths"].sum())],
})
display(cost_breakdown)
print("Test contracts remain unused:", len(test))
print("Feature-preparation timing is omitted here and included in the later full benchmark.")

## 5. Optional extension: average several nearby references

A $k$-nearest-neighbor predictor averages the simulated normalized prices of the $k$ closest references:

$$\widehat y_k(x)=\frac1k\sum_{j\in\mathcal N_k(x)}\widetilde y_j.$$

With $k=1$, this is our nearest-reference rule. Averaging can reduce the influence of one reference's Monte Carlo noise. If reference simulation errors are independent and have variances $s_j^2$, the simulation-noise variance of the average is approximately $k^{-2}\sum_{j\in\mathcal N_k}s_j^2$.

The price-mismatch term may grow as more distant references enter the average. The balance depends on the response function, reference coverage, and simulation budget. A larger $k$ is not automatically better.

The worked example uses a fixed illustrative choice of $k=5$. Any choice based on validation results must be documented and then frozen before the final test. $k$NN is a prediction rule; it does not by itself learn the distance weights.

In [ ]:
pred_knn5, neighbor_knn5 = nearest_predict(Zreference, y_reference, Zvalidation, k=5)
knn_metrics = score_predictions(y_validation_mc, pred_knn5, premium=validation_premiums)
knn_comparison = pd.DataFrame([
    {"method": "Scaled 1NN", "k": 1,
     "RMSE_pp_of_premium": 100 * score_predictions(y_validation_mc, pred_scaled)["rmse"]},
    {"method": "Scaled kNN", "k": 5,
     "RMSE_pp_of_premium": 100 * knn_metrics["rmse"]},
])
display(knn_comparison)
print("These are validation comparisons; no final test score has been calculated.")

## Reading and discussion

| Source | This week's task | Question to answer |
|---|---|---|
| Ben Mingbin Feng, Zhenni Tan, and Jiayi Zheng (2020), [*Efficient Simulation Designs for Valuation of Large Variable Annuity Portfolios*](https://doi.org/10.1080/10920277.2019.1685394), *North American Actuarial Journal*, 24(2), 275–289. [Author PDF](https://www.math.uwaterloo.ca/~mbfeng/papers/2020_VAPort.pdf). | Read the overview of the three stages, skim Section 3, and focus on the CSM predictor in Section 5, especially Equation (12). Leave optimized simulation allocation for an optional later reading. About 30 minutes. | Which object is represented by a simulated contract, and why does scaling by contract size matter? Identify one simplification in our notebook. |
| Guojun Gan and X. Sheldon Lin (2015), [*Valuation of large variable annuity portfolios under nested simulation: A functional data approach*](https://doi.org/10.1016/j.insmatheco.2015.02.007), *Insurance: Mathematics and Economics*, 62, 138–150. | Read the abstract and introduction for the computational motivation. We do not reproduce nested simulation or functional kriging. About 15 minutes. | Why can pricing only a representative subset save simulation work, and what approximation problem does that create? |
| Hao Quan and Ben (Mingbin) Feng (2026), *Warm-Started Projected Robbins–Monro for Portfolio-Scale Variable Annuity Fee Setting*. Mentor-provided working manuscript. | Optional: read the intuitive description of using a similar predecessor contract to initialize a new problem. | What is transferred in that fee-setting workflow, and what do we transfer in this fixed-fee notebook? |

**Submit:** answer the two core questions, and write a paragraph distinguishing input similarity, price similarity, and a model that predicts prices. You are not required to read the mentor's contract-similarity manuscript.

## Coding homework 1 — Compare three feature conventions fairly

Add a `StandardScaler` fitted **only** on the candidate pool. Compare raw, min–max, and standard-deviation scaling with $k=1$, keeping the selected reference IDs and their simulation labels fixed.

For each convention, report RMSE, MAE, and the 95th percentile of absolute validation discrepancy. Inspect one query whose chosen reference changes. Explain what each scaling convention assumes, and whether the observed ranking is strong enough to generalize beyond this split and simulation seed.

Do not resimulate labels, call `gmab_exact`, fit a scaler on validation/test data, or inspect the test scores. No new simulation work is needed for this exercise.

In [ ]:
# Homework starter: this reproduces the existing min-max baseline.
# Add StandardScaler from sklearn.preprocessing, fitted on candidate features only.
hw_scaling_pred, hw_scaling_neighbors = nearest_predict(
    Zreference, y_reference, Zvalidation, k=1
)
hw_scaling_metrics = score_predictions(y_validation_mc, hw_scaling_pred)
display(pd.DataFrame([{
    "scaling": "min-max",
    "RMSE_pp_of_premium": 100 * hw_scaling_metrics["rmse"],
    "MAE_pp_of_premium": 100 * hw_scaling_metrics["mae"],
    "P95_absolute_error_pp": 100 * hw_scaling_metrics["p95_abs"],
}]))
# Add the raw and standard-deviation-scaled rows and one neighbor inspection here.

## Coding homework 2 — Study the effect of the number of neighbors

Using the same min–max features and reference labels, compare $k\in\{1,3,5,9\}$. Report validation RMSE and signed bias and make a plot against $k$. Give a provisional recommendation and explain the noise-versus-price-mismatch tradeoff.

Then write down exactly what you would freeze before evaluating the final test pool: the split, feature transformation, representative selection rule and seed, path budget, and neighbor count. Your recommendation is a validation-based development result, not a final test result.

**Optional sensitivity check:** simulate the same references once more with a different reference seed, keeping the validation labels fixed. Compare whether your qualitative conclusion about $k$ persists. Report the extra simulation work if you do this.

In [ ]:
# Homework starter: extend the list of k values without generating new labels.
hw_k_values = [1]
hw_k_rows = []
for hw_k in hw_k_values:
    hw_pred, _ = nearest_predict(Zreference, y_reference, Zvalidation, k=hw_k)
    hw_metrics = score_predictions(y_validation_mc, hw_pred)
    hw_k_rows.append({
        "k": hw_k,
        "RMSE_pp_of_premium": 100 * hw_metrics["rmse"],
        "signed_bias_pp": 100 * hw_metrics["bias"],
    })
display(pd.DataFrame(hw_k_rows))
# Add the other k values, the plot, and a written provisional recommendation here.

## Weekly deliverable and checklist

Submit your completed notebook, both homework comparisons, and the reading responses. Add a 200-word research note: **What has to be true for nearest-reference pricing to work, and which part of that assumption can we improve next?** Use the actual outputs; do not state an expected model ranking as an observed result.

- [ ] I split contract IDs before fitting transformations and generating prediction labels.
- [ ] I fitted each scaler using candidate features only.
- [ ] I excluded premium from normalized-price distance and rescaled query prices correctly.
- [ ] I held reference IDs, reference labels, and validation labels fixed across the main comparison.
- [ ] I can separate reference simulation noise from price mismatch between contracts.
- [ ] I can interpret validation RMSE without calling Monte Carlo labels exact truths.
- [ ] I recorded simulation path work and identified which timings were included.
- [ ] I did not generate, tune on, or evaluate test labels.
- [ ] I did not use the analytical audit function for training or validation.
- [ ] I restarted the kernel and ran every cell from top to bottom.

**Three-student meeting roles:** preprocessing/split checker, predictor explainer, and results/uncertainty reviewer. Next week we replace random reference selection with an unsupervised clustering compressor.